# Deps

In [1]:
!pip install checklist-plus --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. T

In [2]:
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Load our Example Embedding Model

In [ ]:
# prompt: import bert base embedding model from hugging face

from transformers import BertModel, BertTokenizer
import torch

# Load pre-trained model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Example usage:
text = "This is a sample sentence."
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)

# The output is a dictionary containing 'last_hidden_state' and 'pooler_output'
last_hidden_states = output.last_hidden_state
pooler_output = output.pooler_output

print("Last hidden states shape:", last_hidden_states.shape)
print("Pooler output shape:", pooler_output.shape)


In [2]:
# prompt: now compuse cosine similarity between three texts (1 relevant pair and 1 irrelevant text)

from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertModel, BertTokenizer
import torch

# Load pre-trained model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_embedding(text):
  encoded_input = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
  with torch.no_grad():
    output = model(**encoded_input)
  # Use the CLS token embedding as the sentence embedding
  sentence_embedding = output.last_hidden_state[:, 0, :].numpy()
  return sentence_embedding

# Define the texts
text1 = "The weather is nice today."
text2 = "It's a beautiful day outside."
text3 = "The stock market crashed yesterday."

# Get embeddings for each text
embedding1 = get_embedding(text1)
embedding2 = get_embedding(text2)
embedding3 = get_embedding(text3)

# Calculate cosine similarity
similarity_1_2 = cosine_similarity(embedding1, embedding2)[0][0]
similarity_1_3 = cosine_similarity(embedding1, embedding3)[0][0]
similarity_2_3 = cosine_similarity(embedding2, embedding3)[0][0]

print(f"Cosine similarity between text1 and text2: {similarity_1_2:.4f}")
print(f"Cosine similarity between text1 and text3: {similarity_1_3:.4f}")
print(f"Cosine similarity between text2 and text3: {similarity_2_3:.4f}")

Cosine similarity between text1 and text2: 0.9522
Cosine similarity between text1 and text3: 0.8442
Cosine similarity between text2 and text3: 0.8108


# Set OPENAI API KEY

In [3]:
import os
os.environ['OPENAI_API_KEY'] = "your-api-key"


In [4]:
import checklist_plus
from checklist_plus.editor import Editor
from checklist_plus.perturb import LLMPerturb

In [5]:
checklist_plus.__version__

'0.2.0'

In [6]:
llm_editor = Editor(
             use_llm=True,
            model_name='gpt-4o-mini')

## Generate Examples data

In [10]:
ret = llm_editor.template('The football game was very good, I especially liked {mask}', context="different experiences in football games", remove_duplicates=True, n_completions=100)
original_texts = ret.data

In [11]:
original_texts = list(set(original_texts))
print(original_texts[:5])

["The football game was very good, I especially liked team's cohesion", 'The football game was very good, I especially liked match strategy', "The football game was very good, I especially liked goalkeeper's save", 'The football game was very good, I especially liked match result', 'The football game was very good, I especially liked matchday experience']


In [12]:
len(original_texts)

90

## Paraphrase Example Data

In [13]:
ret = llm_editor.paraphrase_llm(original_texts, n_paraphrases=1, length_preference='similar')

In [14]:
paraphrased_texts = ret.data

In [15]:
assert len(paraphrased_texts) == len(original_texts)

In [16]:
paraphrased_texts[:5]

["The match was excellent, and I particularly appreciated the team's unity.",
 'The soccer match was excellent; I particularly enjoyed the tactical approach to the game.',
 "The soccer match was quite impressive; I particularly enjoyed the goalkeeper's remarkable save.",
 'The soccer match was quite enjoyable; I particularly appreciated the outcome of the game.',
 'The soccer match was excellent; I particularly enjoyed the experience of being at the game day.']

## Negate Example Data

In [17]:
perturb = LLMPerturb()

In [18]:
ret = perturb.add_negation_llm(original_texts, n_variations=1)

In [19]:
negated_texts = [x[0] for x in ret]

In [20]:
negated_texts[:5]

["The football game was not good at all; I didn't like the team's lack of cohesion.",
 'The football game was not very good; I did not like the match strategy at all.',
 "The football game was not very good; I did not particularly like the goalkeeper's save.",
 'The football game was not very good; I did not especially like the match result.',
 'The football game was not very good; I did not particularly enjoy the matchday experience.']

# Perform Simple MFT test


In [21]:
from checklist_plus.test_types import MFT, INV, DIR
from checklist_plus.expect import Expect

In [22]:
# expect original text is more similar to the paraphrased one
def similar_paraphrase(x, pred, conf, label=None, meta=None):
    return pred == 0
expect_fn = Expect.single(similar_paraphrase)

In [23]:
test = MFT(list(zip(original_texts, paraphrased_texts, negated_texts)), expect=expect_fn, name='Simple negation',
           capability='Negation', description='Very simple negations.')

In [24]:
import numpy as np
def get_cosine_similarities(data):
  similarities = []
  for original, paraphrased, negated in data:
    original_embedding = get_embedding(original)
    paraphrased_embedding = get_embedding(paraphrased)
    negated_embedding = get_embedding(negated)

    sim_paraphrased = cosine_similarity(original_embedding, paraphrased_embedding)[0][0]
    sim_negated = cosine_similarity(original_embedding, negated_embedding)[0][0]

    similarities.append([sim_paraphrased, sim_negated])
  similarities = np.array(similarities)
  return np.argmax(similarities, axis=-1), similarities

cosine_sims = get_cosine_similarities(list(zip(original_texts, paraphrased_texts, negated_texts))[:5])

In [25]:
print(cosine_sims[:5])


(array([1, 1, 1, 1, 0]), array([[0.9132893 , 0.92996585],
       [0.9589352 , 0.9602138 ],
       [0.8971145 , 0.91686565],
       [0.90862095, 0.9227729 ],
       [0.9465144 , 0.9420063 ]], dtype=float32))


In [26]:
test.run(get_cosine_similarities)

Predicting 90 examples


In [27]:
# bert-base-uncased is not sensitive to negations
test.summary()

Test cases:      90
Fails (rate):    76 (84.4%)

Example fails:
0.9 ("The football game was very good, I especially liked player's commitment", 'The soccer match was excellent, and I particularly appreciated the dedication of the players.', "The football game was not very good; I did not like the players' commitment at all.")
----
0.9 ("The football game was very good, I especially liked team's history", "The soccer match was quite impressive, and I particularly appreciated the team's background.", "The football game was not very good; I did not especially like the team's history.")
----
0.9 ("The football game was very good, I especially liked team's spirit", "The match was excellent, and I particularly appreciated the team's enthusiasm.", "The football game was not very good; I did not especially like the team's spirit.")
----


# Let's test whether Embedding model understands sizes in our marketplace

## Generate data

In [28]:
ret = llm_editor.template('red {mask} shoes US 11 size ', context="different shoe brands", remove_duplicates=True, n_completions=100)
user_queries = ret.data

In [29]:
user_queries = list(set(user_queries))
print(user_queries[:5])

['red Salomon shoes US 11 size ', 'red Sorel shoes US 11 size ', 'red Ariat shoes US 11 size ', "red Dr. Martens's shoes US 11 size ", "red Superga's shoes US 11 size "]


In [30]:
len(user_queries)

91

In [31]:
ret = llm_editor.paraphrase_llm(user_queries, n_paraphrases=1, length_preference='longer')

In [32]:
item_titles = ret.data
print(item_titles[:5])

['Size 11 US red sneakers from the brand Salomon.', 'A pair of red Sorel footwear in a U.S. size 11.', 'A pair of size 11 red shoes from the brand Ariat, designed for men in the United States.', 'A pair of red Dr. Martens shoes in size 11 US.', 'A pair of red Superga sneakers in size US 11.']


In [33]:
assert len(item_titles) == len(user_queries)

In [34]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [35]:
ret = LLMPerturb.perturb(list(nlp.pipe(item_titles)), LLMPerturb.change_number, n=1, keep_original=False)
item_titles_negative = [o[0] for o in ret.data]

In [36]:
assert len(item_titles) == len(item_titles_negative)

In [37]:
print(user_queries[:3], item_titles_negative[:3], item_titles[:3])

['red Salomon shoes US 11 size ', 'red Sorel shoes US 11 size ', 'red Ariat shoes US 11 size '] ['Size 9 US red sneakers from the brand Salomon.', 'A pair of red Sorel footwear in a U.S. size 10.', 'A pair of size 8 red shoes from the brand Ariat, designed for men in the United States.'] ['Size 11 US red sneakers from the brand Salomon.', 'A pair of red Sorel footwear in a U.S. size 11.', 'A pair of size 11 red shoes from the brand Ariat, designed for men in the United States.']


## Run the test

In [38]:
test = MFT(list(zip(user_queries, item_titles, item_titles_negative)), expect=expect_fn, name='Size change',
           capability='Size', description='Very simple size understanding check.')

In [39]:
test.run(get_cosine_similarities)

Predicting 91 examples


In [41]:
# bert-base-uncased fails 33% of the times even when texts are identical
test.summary()

Test cases:      91
Fails (rate):    30 (33.0%)

Example fails:
0.9 ("red Etnies's shoes US 11 size ", 'A pair of red Etnies shoes in size US 11.', 'A pair of red Etnies shoes in size US 9.')
----
0.9 ('red DC Shoes shoes US 11 size ', 'A pair of red DC Shoes in size 11 for men in the United States.', 'A pair of red DC Shoes in size 10 for men in the United States.')
----
0.8 ('red Under Armour shoes US 11 size ', 'Crimson-colored Under Armour footwear in a size 11 for men in the United States.', 'Crimson-colored Under Armour footwear in a size 9 for men in the United States.')
----


# Let's test whether Embedding model understands colors
We want embeddings with relevant colors be closer

##Generate Examples data

In [42]:
ret = llm_editor.template('{mask} phone color of {mask}', context="mask1 is for brands and mask2 is for colors", remove_duplicates=True, n_completions=100)
user_queries = ret.data

In [43]:
user_queries = list(set(user_queries))
print(user_queries[:5])

['Meizu phone color of lavender', 'BlackBerry phone color of citrus', 'Alcatel phone color of coral', 'Galaxy phone color of slate', 'OnePlus phone color of white']


In [44]:
len(user_queries)

79

##Paraphrase colors only

In [45]:
examples = [
    ("Motorola phone color crimson", "Motorola phone of a rich deep red colour"),
    ("Sharp phone color coffee", "Sharp phone of a deep brown colour"),
    ("Samsung phone color white", "Samsung phone of a pure bright colour"),
]
ret = llm_editor.paraphrase_llm(user_queries, n_paraphrases=1, length_preference='similar', context="only colors", examples=examples)
item_titles = ret.data
print(item_titles[:5])

['Meizu phone in a soft purple hue', 'BlackBerry phone in a vibrant citrus hue', 'Alcatel phone in a vibrant coral hue', 'Galaxy phone in a shade of grayish blue', 'OnePlus phone in a pristine shade of white']


##Remove colors for negative samples

In [46]:
ret = llm_editor.detect_and_mask_entities_llm(user_queries, entity_type="COLORS", mask_token="")

In [51]:
item_titles_no_colors = [o["masked_text"] for o in ret.data if o["contains_entities"]]
data = []
for idx, o in enumerate(ret.data):
    if o["contains_entities"]:
        data.append((user_queries[idx], item_titles[idx], o["masked_text"].replace("color of", "")))

##Run the test

In [56]:
test = MFT(data, expect=expect_fn, name='Color understanding',
           capability='Color', description='Very simple color understanding check.')

In [57]:
test.run(get_cosine_similarities)

Predicting 74 examples


In [58]:
test.summary()

Test cases:      74
Fails (rate):    35 (47.3%)

Example fails:
0.9 ('Huawei phone color of ash', 'Huawei phone in a shade of gray', 'Huawei phone  ')
----
0.9 ('Wiko phone color of cinnamon', 'Wiko phone in a shade of warm brown spice', 'Wiko phone  ')
----
0.9 ('Dell phone color of indigo', 'Dell phone in a shade of deep blue', 'Dell phone  ')
----
